LANID GTPS Preprocess

In [ ]:
import ee
try:
    ee.Initialize(project='remotesensinggi')   # <- your EE project ID
except Exception:
    ee.Authenticate()
    ee.Initialize(project='remotesensinggi')

# -------------------------
# Inputs
# -------------------------
LANID_ASSET = "projects/remotesensinggi/assets/LANID_GPTS_2017"  # <-- change
lanid_fc = ee.FeatureCollection(LANID_ASSET)

cropNE = ee.Image("projects/ee-endalkmamlove/assets/NE")
cropNW = ee.Image("projects/ee-endalkmamlove/assets/NW")
cropSE = ee.Image("projects/ee-endalkmamlove/assets/SE")
cropSW = ee.Image("projects/ee-endalkmamlove/assets/SW")
cropland = ee.ImageCollection([cropNE, cropNW, cropSE, cropSW]).mosaic()

cropland_mask = cropland.gt(0).selfMask()  # adjust if needed

CROPLAND_FRACTION_THRESHOLD = 0.5
NDVI_DIFF_TOL = 0.1
RAINFED_TO_IRRIG_NDVI2017 = 0.6

EXPORT_FOLDER = "Pre Processed GTPs"  # <-- Drive folder
SCALE = 30

# -------------------------
# Landsat NDVI functions
# -------------------------
def preprocess_l8l9_sr(img):
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")

    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 1).eq(0)    # dilated cloud
            .And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus
            .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))  # shadow
            .And(qa.bitwiseAnd(1 << 5).eq(0))) # snow

    return ndvi.updateMask(mask).copyProperties(img, img.propertyNames())

def ndvi_max_apr_oct(year, region_geom):
    start = ee.Date.fromYMD(year, 4, 1)
    end   = ee.Date.fromYMD(year, 10, 31).advance(1, "day")

    l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterDate(start, end)
          .filterBounds(region_geom)
          .map(preprocess_l8l9_sr))

    l9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
          .filterDate(start, end)
          .filterBounds(region_geom)
          .map(preprocess_l8l9_sr))

    return l8.merge(l9).max().rename(f"NDVImax_{year}")

# -------------------------
# Helpers for polygon stats
# -------------------------
def add_cropland_fraction(feat):
    geom = feat.geometry()
    frac = cropland_mask.unmask(0).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=SCALE,
        maxPixels=1e13,
        bestEffort=True
    ).values().get(0)
    return feat.set("cropland_frac", frac)

def add_ndvi_props_factory(stack_img):
    # return a function to map over features using the given image
    def _fn(feat):
        geom = feat.geometry()
        stats = stack_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=SCALE,
            maxPixels=1e13,
            bestEffort=True
        )
        return feat.set({
            "NDVImax_2017": stats.get("NDVImax_2017"),
            "NDVImax_current": stats.get("NDVImax_current")
        })
    return _fn

def relabel(feat):
    irr = ee.Number(feat.get("irrigated"))
    n17 = ee.Number(feat.get("NDVImax_2017"))
    ncu = ee.Number(feat.get("NDVImax_current"))
    diff = n17.subtract(ncu).abs()

    # irrigated==1: keep 1 only if diff <= tol else 0
    new_if_irr = ee.Algorithms.If(diff.lte(NDVI_DIFF_TOL), 1, 0)

    # rainfed==0: if NDVI2017 >= 0.6 => 1 else 0 (and diff rule "keep rainfed" doesn't change value anyway)
    new_if_rainfed = ee.Algorithms.If(n17.gte(RAINFED_TO_IRRIG_NDVI2017), 1, 0)

    new_irrig = ee.Number(ee.Algorithms.If(irr.eq(1), new_if_irr, new_if_rainfed))

    return feat.set({"irrigated_new": new_irrig, "ndvi_diff_abs": diff})

# -------------------------
# State-by-state loop (TIGER)
# -------------------------
states = ee.FeatureCollection("TIGER/2018/States")

# If you only want a subset, set a list of state names:
# TARGET_STATES = ["Delaware", "California"]
TARGET_STATES = None

state_list = states.aggregate_array("NAME").getInfo()
tasks = []

for st_name in state_list:
    if TARGET_STATES and st_name not in TARGET_STATES:
        continue

    st_feat = states.filter(ee.Filter.eq("NAME", st_name)).first()
    st_geom = st_feat.geometry()

    # Filter GTPs to state
    fc_state = lanid_fc.filterBounds(st_geom)

    # Skip empty states quickly
    count = fc_state.size().getInfo()
    if count == 0:
        print("Skipping (no features):", st_name)
        continue

    print(f"{st_name}: {count} features before cropland filter")

    # Cropland fraction filter
    fc_state = fc_state.map(add_cropland_fraction) \
                       .filter(ee.Filter.gte("cropland_frac", CROPLAND_FRACTION_THRESHOLD))

    count2 = fc_state.size().getInfo()
    print(f"{st_name}: {count2} after cropland filter")

    if count2 == 0:
        print("Skipping (no features after cropland filter):", st_name)
        continue

    # NDVI images clipped to state for efficiency
    ndvi2017 = ndvi_max_apr_oct(2017, st_geom).rename("NDVImax_2017")
    ndvi2023 = ndvi_max_apr_oct(2023, st_geom).rename("NDVImax_current")
    # (If you prefer max of 2023/2024, compute ndvi2024 and max them)

    stack = ndvi2017.addBands(ndvi2023).clip(st_geom)

    # Add NDVI properties and relabel
    fc_state = fc_state.map(add_ndvi_props_factory(stack)) \
                       .filter(ee.Filter.notNull(["NDVImax_2017", "NDVImax_current"])) \
                       .map(relabel)

    # Export per state
    export_desc = f"LANID_GTPs_relab_{st_name.replace(' ', '_')}"
    task = ee.batch.Export.table.toDrive(
        collection=fc_state,
        description=export_desc,
        folder=EXPORT_FOLDER,
        fileFormat="SHP"
    )
    task.start()
    tasks.append(task)

print(f"Started {len(tasks)} export tasks. Check the EE Tasks tab.")


Skipping (no features): United States Virgin Islands
Skipping (no features): Commonwealth of the Northern Mariana Islands
Skipping (no features): Guam
Skipping (no features): American Samoa
Skipping (no features): Puerto Rico
Rhode Island: 12 features before cropland filter
Rhode Island: 6 after cropland filter
New Hampshire: 29 features before cropland filter
New Hampshire: 25 after cropland filter
Vermont: 37 features before cropland filter
Vermont: 31 after cropland filter
Connecticut: 29 features before cropland filter
Connecticut: 28 after cropland filter
Maine: 52 features before cropland filter
Maine: 41 after cropland filter
Massachusetts: 35 features before cropland filter
Massachusetts: 13 after cropland filter
New Jersey: 76 features before cropland filter
New Jersey: 71 after cropland filter
Pennsylvania: 204 features before cropland filter
Pennsylvania: 181 after cropland filter
New York: 156 features before cropland filter
New York: 140 after cropland filter
Illinois: 773

GMIE GTPs Preprocess

In [ ]:
import ee
try:
    ee.Initialize(project='remotesensinggi')   # <- your EE project ID
except Exception:
    ee.Authenticate()
    ee.Initialize(project='remotesensinggi')

# -------------------------
# Inputs
# -------------------------
LANID_ASSET = "projects/remotesensinggi/assets/GMIE_USA"  # <-- change
lanid_fc = ee.FeatureCollection(LANID_ASSET)

cropNE = ee.Image("projects/ee-endalkmamlove/assets/NE")
cropNW = ee.Image("projects/ee-endalkmamlove/assets/NW")
cropSE = ee.Image("projects/ee-endalkmamlove/assets/SE")
cropSW = ee.Image("projects/ee-endalkmamlove/assets/SW")
cropland = ee.ImageCollection([cropNE, cropNW, cropSE, cropSW]).mosaic()

cropland_mask = cropland.gt(0).selfMask()  # adjust if needed

CROPLAND_FRACTION_THRESHOLD = 0.5
NDVI_DIFF_TOL = 0.1
RAINFED_TO_IRRIG_NDVI2017 = 0.6

EXPORT_FOLDER = "Pre Processed GTPs"  # <-- Drive folder
SCALE = 30

# -------------------------
# Landsat NDVI functions
# -------------------------
def preprocess_l8l9_sr(img):
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")

    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 1).eq(0)    # dilated cloud
            .And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus
            .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))  # shadow
            .And(qa.bitwiseAnd(1 << 5).eq(0))) # snow

    return ndvi.updateMask(mask).copyProperties(img, img.propertyNames())

def ndvi_max_apr_oct(year, region_geom):
    start = ee.Date.fromYMD(year, 4, 1)
    end   = ee.Date.fromYMD(year, 10, 31).advance(1, "day")

    l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterDate(start, end)
          .filterBounds(region_geom)
          .map(preprocess_l8l9_sr))

    l9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
          .filterDate(start, end)
          .filterBounds(region_geom)
          .map(preprocess_l8l9_sr))

    return l8.merge(l9).max().rename(f"NDVImax_{year}")

# -------------------------
# Helpers for polygon stats
# -------------------------
def add_cropland_fraction(feat):
    geom = feat.geometry()
    frac = cropland_mask.unmask(0).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=SCALE,
        maxPixels=1e13,
        bestEffort=True
    ).values().get(0)
    return feat.set("cropland_frac", frac)

def add_ndvi_props_factory(stack_img):
    # return a function to map over features using the given image
    def _fn(feat):
        geom = feat.geometry()
        stats = stack_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=SCALE,
            maxPixels=1e13,
            bestEffort=True
        )
        return feat.set({
            "NDVImax_2017": stats.get("NDVImax_2017"),
            "NDVImax_current": stats.get("NDVImax_current")
        })
    return _fn

def relabel(feat):
    irr = ee.Number(feat.get("irrigated"))
    n17 = ee.Number(feat.get("NDVImax_2017"))
    ncu = ee.Number(feat.get("NDVImax_current"))
    diff = n17.subtract(ncu).abs()

    # irrigated==1: keep 1 only if diff <= tol else 0
    new_if_irr = ee.Algorithms.If(diff.lte(NDVI_DIFF_TOL), 1, 0)

    # rainfed==0: if NDVI2017 >= 0.6 => 1 else 0 (and diff rule "keep rainfed" doesn't change value anyway)
    new_if_rainfed = ee.Algorithms.If(n17.gte(RAINFED_TO_IRRIG_NDVI2017), 1, 0)

    new_irrig = ee.Number(ee.Algorithms.If(irr.eq(1), new_if_irr, new_if_rainfed))

    return feat.set({"irrigated_new": new_irrig, "ndvi_diff_abs": diff})

# -------------------------
# State-by-state loop (TIGER)
# -------------------------
states = ee.FeatureCollection("TIGER/2018/States")

# If you only want a subset, set a list of state names:
# TARGET_STATES = ["Delaware", "California"]
TARGET_STATES = None

state_list = states.aggregate_array("NAME").getInfo()
tasks = []

for st_name in state_list:
    if TARGET_STATES and st_name not in TARGET_STATES:
        continue

    st_feat = states.filter(ee.Filter.eq("NAME", st_name)).first()
    st_geom = st_feat.geometry()

    # Filter GTPs to state
    fc_state = lanid_fc.filterBounds(st_geom)

    # Skip empty states quickly
    count = fc_state.size().getInfo()
    if count == 0:
        print("Skipping (no features):", st_name)
        continue

    print(f"{st_name}: {count} features before cropland filter")

    # Cropland fraction filter
    fc_state = fc_state.map(add_cropland_fraction) \
                       .filter(ee.Filter.gte("cropland_frac", CROPLAND_FRACTION_THRESHOLD))

    count2 = fc_state.size().getInfo()
    print(f"{st_name}: {count2} after cropland filter")

    if count2 == 0:
        print("Skipping (no features after cropland filter):", st_name)
        continue

    # NDVI images clipped to state for efficiency
    ndvi2017 = ndvi_max_apr_oct(2019, st_geom).rename("NDVImax_2017")
    ndvi2023 = ndvi_max_apr_oct(2023, st_geom).rename("NDVImax_current")
    # (If you prefer max of 2023/2024, compute ndvi2024 and max them)

    stack = ndvi2017.addBands(ndvi2023).clip(st_geom)

    # Add NDVI properties and relabel
    fc_state = fc_state.map(add_ndvi_props_factory(stack)) \
                       .filter(ee.Filter.notNull(["NDVImax_2017", "NDVImax_current"])) \
                       .map(relabel)

    # Export per state
    export_desc = f"GMIE_GTPs_relab_{st_name.replace(' ', '_')}"
    task = ee.batch.Export.table.toDrive(
        collection=fc_state,
        description=export_desc,
        folder=EXPORT_FOLDER,
        fileFormat="SHP"
    )
    task.start()
    tasks.append(task)

print(f"Started {len(tasks)} export tasks. Check the EE Tasks tab.")


Skipping (no features): United States Virgin Islands
Skipping (no features): Commonwealth of the Northern Mariana Islands
Skipping (no features): Guam
Skipping (no features): American Samoa
Skipping (no features): Puerto Rico
Skipping (no features): Rhode Island
Skipping (no features): New Hampshire
Skipping (no features): Vermont
Skipping (no features): Connecticut
Skipping (no features): Maine
Skipping (no features): Massachusetts
New Jersey: 5 features before cropland filter
New Jersey: 4 after cropland filter
Skipping (no features): Pennsylvania
Skipping (no features): New York
Illinois: 498 features before cropland filter
Illinois: 485 after cropland filter
Wisconsin: 551 features before cropland filter
Wisconsin: 501 after cropland filter
Ohio: 12 features before cropland filter
Ohio: 12 after cropland filter
Michigan: 421 features before cropland filter
Michigan: 398 after cropland filter
Indiana: 494 features before cropland filter
Indiana: 468 after cropland filter
Minnesota: 

USGS GTPS

In [ ]:
import ee
try:
    ee.Initialize(project='remotesensinggi')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='remotesensinggi')

# -------------------------
# Inputs
# -------------------------
USGS_ASSET = "projects/remotesensinggi/assets/USGS_GTPS"
gtp_fc = ee.FeatureCollection(USGS_ASSET)

# GLAD cropland mosaic
cropNE = ee.Image("projects/ee-endalkmamlove/assets/NE")
cropNW = ee.Image("projects/ee-endalkmamlove/assets/NW")
cropSE = ee.Image("projects/ee-endalkmamlove/assets/SE")
cropSW = ee.Image("projects/ee-endalkmamlove/assets/SW")
cropland = ee.ImageCollection([cropNE, cropNW, cropSE, cropSW]).mosaic()
cropland_mask = cropland.gt(0).selfMask()  # adjust if needed

NDVI_DIFF_TOL = 0.1
EXPORT_FOLDER = "Pre Processed GTPs"
SCALE = 30

# -------------------------
# Basin boundary
# -------------------------
# Option A: Use a UCRB polygon you uploaded as an EE asset (recommended if you have it)
# UCRB = ee.FeatureCollection("projects/remotesensinggi/assets/UCRB_boundary").geometry()

# Option B: Use WBD HUC2 if available in your EE catalog:
# Upper Colorado River Basin is typically HUC2 = "14"
# If this ID errors, tell me and I’ll switch to an alternative basin layer.
UCRB = (ee.FeatureCollection("USGS/WBD/2017/HUC02")
        .filter(ee.Filter.eq("huc2", "14"))
        .geometry())

# -------------------------
# Landsat NDVI helpers
# -------------------------
def preprocess_l8l9_sr(img):
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 1).eq(0)
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0)))
    return ndvi.updateMask(mask)

def preprocess_l57_sr(img):
    red = img.select("SR_B3").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 1).eq(0)
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0)))
    return ndvi.updateMask(mask)

def ndvi_max_apr_oct_l89(year, region_geom):
    start = ee.Date.fromYMD(year, 4, 1)
    end   = ee.Date.fromYMD(year, 10, 31).advance(1, "day")
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterDate(start, end).filterBounds(region_geom).map(preprocess_l8l9_sr)
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterDate(start, end).filterBounds(region_geom).map(preprocess_l8l9_sr)
    return l8.merge(l9).max().rename(f"NDVImax_{year}")

def ndvi_max_apr_oct_l57(year, region_geom):
    start = ee.Date.fromYMD(year, 4, 1)
    end   = ee.Date.fromYMD(year, 10, 31).advance(1, "day")
    l5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filterDate(start, end).filterBounds(region_geom).map(preprocess_l57_sr)
    l7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filterDate(start, end).filterBounds(region_geom).map(preprocess_l57_sr)
    return l5.merge(l7).max().rename(f"NDVImax_{year}")

def baseline_mean_of_ndvi_max_2007_2010(region_geom):
    imgs = ee.ImageCollection([
        ndvi_max_apr_oct_l57(2007, region_geom),
        ndvi_max_apr_oct_l57(2008, region_geom),
        ndvi_max_apr_oct_l57(2009, region_geom),
        ndvi_max_apr_oct_l57(2010, region_geom),
    ])
    return imgs.mean().rename("NDVImax_mean_2007_2010")

# -------------------------
# Feature ops (POINT-based, cheap)
# -------------------------
def to_centroid(feat):
    return ee.Feature(feat.geometry().centroid(1), feat.toDictionary())

def add_cropland_pt(feat):
    band = cropland_mask.bandNames().get(0)
    samp = cropland_mask.unmask(0).sample(
        region=feat.geometry(),
        scale=SCALE,
        numPixels=1,
        geometries=False
    ).first()

    val = ee.Algorithms.If(samp, ee.Number(ee.Feature(samp).get(band)), 0)
    return feat.set("cropland_pt", val)


def add_ndvi_pt(stack_img):
    def _fn(feat):
        samp = stack_img.sample(
            region=feat.geometry(),
            scale=SCALE,
            numPixels=1,
            geometries=False
        ).first()

        base = ee.Algorithms.If(samp, ee.Feature(samp).get("NDVImax_mean_2007_2010"), None)
        cur  = ee.Algorithms.If(samp, ee.Feature(samp).get("NDVImax_2023"), None)

        return feat.set({
            "NDVImax_mean_2007_2010": base,
            "NDVImax_2023": cur
        })
    return _fn


def relabel(feat):
    irr = ee.Number(feat.get("irrigated"))
    base = ee.Number(feat.get("NDVImax_mean_2007_2010"))
    cur  = ee.Number(feat.get("NDVImax_2023"))
    diff = base.subtract(cur).abs()
    new_if_irr = ee.Algorithms.If(diff.lte(NDVI_DIFF_TOL), 1, 0)
    new_irrig = ee.Number(ee.Algorithms.If(irr.eq(1), new_if_irr, 0))
    return feat.set({"irrigated_new": new_irrig, "ndvi_diff_abs": diff})

# -------------------------
# Counties that intersect UCRB (and optional UCRB states only)
# -------------------------
counties = ee.FeatureCollection("TIGER/2018/Counties").filterBounds(UCRB)

# If you want only UCRB states (typical: CO, UT, WY, NM, AZ),
# filter by STATEFP codes:
# CO=08, UT=49, WY=56, NM=35, AZ=04
UCRB_STATEFPS = ["08", "49", "56", "35", "04"]
counties = counties.filter(ee.Filter.inList("STATEFP", UCRB_STATEFPS))

# Pull a SMALL list of county GEOIDs to loop in Python
# (this is safe because UCRB counties count is modest)
county_ids = counties.aggregate_array("GEOID").getInfo()
print("UCRB counties to export:", len(county_ids))

tasks = []

for geoid in county_ids:
    county = counties.filter(ee.Filter.eq("GEOID", geoid)).first()
    county_geom = county.geometry()

    # Filter your GTPs to county, centroid them
    fc = gtp_fc.filterBounds(county_geom).map(to_centroid)

    # Cheap cropland filter by centroid
    fc = fc.map(add_cropland_pt).filter(ee.Filter.eq("cropland_pt", 1))

    # Build NDVI stack clipped to county
    baseline = baseline_mean_of_ndvi_max_2007_2010(county_geom)
    ndvi2023 = ndvi_max_apr_oct_l89(2023, county_geom).rename("NDVImax_2023")
    stack = baseline.addBands(ndvi2023).clip(county_geom)

    # Sample NDVI at point + relabel
    fc = (fc.map(add_ndvi_pt(stack))
            .filter(ee.Filter.notNull(["NDVImax_mean_2007_2010", "NDVImax_2023"]))
            .map(relabel))

    # Export
    desc = f"USGS_GTPS_UCRB_{geoid}_070810_vs_2023"
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=desc,
        folder=EXPORT_FOLDER,
        fileFormat="CSV"   # <-- strongly recommend CSV for speed
        # change to "SHP" if you really need shapefile
    )
    task.start()
    tasks.append(task)

print(f"Started {len(tasks)} county exports. Check Earth Engine Tasks tab.")


UCRB counties to export: 62
Started 62 county exports. Check Earth Engine Tasks tab.


South America GMIE

In [ ]:
import ee
try:
    ee.Initialize(project='remotesensinggi')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='remotesensinggi')

# -------------------------
# Inputs
# -------------------------
GTP_ASSET = "projects/remotesensinggi/assets/GMIE_South_America"
gtp_fc = ee.FeatureCollection(GTP_ASSET)

EXPORT_FOLDER = "Pre Processed GTPs"
SCALE = 30
NDVI_DIFF_TOL = 0.1

# Property name in your GMIE shapefile that contains the class label (CHANGE ME if different)
LABEL_FIELD = "irrigated"   # e.g., "irrigated" or "label" or "class"

# -------------------------
# Province season windows
# Format: ("-MM-DD","-MM-DD")
# Cross-year seasons handled (e.g., Nov->Mar)
# -------------------------
ARGENTINA_IRRIG_WINDOW = {
  "Argentina_Buenos Aires":           ("-11-01","-03-31"),
  "Argentina_Buenos Aires D.f.":      ("-11-01","-03-31"),
  "Argentina_Cordoba":                ("-11-01","-03-31"),
  "Argentina_Santa Fe":               ("-11-01","-03-31"),
  "Argentina_Entre Rios":             ("-11-01","-03-31"),
  "Argentina_Corrientes":             ("-12-01","-02-28"),
  "Argentina_Chaco":                  ("-11-01","-03-31"),
  "Argentina_Formosa":                ("-12-01","-02-28"),
  "Argentina_Misiones":               ("-12-01","-02-28"),
  "Argentina_Santiago Del Estero":    ("-11-01","-03-31"),
  "Argentina_La Pampa":               ("-11-01","-03-31"),
  "Argentina_San_Luis":               ("-11-01","-03-31"),
  "Argentina_Rio_Negro":              ("-10-01","-03-31"),
  "Argentina_Tucuman":                ("-11-01","-03-31"),
  "Argentina_Jujuy":                  ("-11-01","-03-31"),
  "Argentina_Salta":                  ("-11-01","-03-31"),
  "Argentina_Mendoza":                ("-10-01","-03-31"),
  "Argentina_San Juan":               ("-10-01","-03-31"),
  "Argentina_La Rioja":               ("-10-01","-03-31"),
  "Argentina_Catamarca":              ("-10-01","-03-31"),
  "Argentina_Neuquen":                ("-12-01","-02-28"),
  "Argentina_Chubut":                 ("-12-01","-02-28"),
  "Argentina_Santa Cruz":             ("-12-01","-02-28"),
  "Argentina_Tierra Del Fuego":       ("-12-15","-02-15"),
}

BRAZIL_IRRIG_WINDOW = {
  "Brazil_Rondonia":               ("-08-01","-11-30"),
  "Brazil_Roraima":                ("-12-01","-03-31"),
  "Brazil_Para":                   ("-07-01","-11-30"),
  "Brazil_Tocantins":              ("-05-01","-10-31"),
  "Brazil_Rio Grande Do Norte":    ("-08-01","-11-30"),
  "Brazil_Paraiba":                ("-08-01","-11-30"),
  "Brazil_Pernambuco":             ("-08-01","-11-30"),
  "Brazil_Piaui":                  ("-08-01","-11-30"),
  "Brazil_Sergipe":                ("-08-01","-11-30"),
  "Brazil_Mato Grosso Do Sul":     ("-05-01","-09-30"),
  # Add the rest of Brazil states here if you want (recommended)
}

# Merge windows into one dict
IRRIG_WINDOW = {}
IRRIG_WINDOW.update(ARGENTINA_IRRIG_WINDOW)
IRRIG_WINDOW.update(BRAZIL_IRRIG_WINDOW)

# -------------------------
# GAUL ADM1 boundaries
# -------------------------
adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Helper to build a GAUL filter key from country+adm1.
# GAUL fields commonly include:
# - ADM0_NAME (country)
# - ADM1_NAME (admin1)
# We'll use those.
def get_adm1_geom(country_name, adm1_name):
    f = (adm1
         .filter(ee.Filter.eq("ADM0_NAME", country_name))
         .filter(ee.Filter.eq("ADM1_NAME", adm1_name))
         .first())
    return ee.Algorithms.If(f, ee.Feature(f).geometry(), None)

# -------------------------
# Landsat NDVI (L8/L9)
# -------------------------
def preprocess_l8l9_sr(img):
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")

    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 1).eq(0)
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0)))

    return ndvi.updateMask(mask).copyProperties(img, img.propertyNames())

def season_dates_for_year(year, start_sfx, end_sfx):
    """
    start_sfx/end_sfx are like "-11-01", "-03-31"
    If end < start (cross-year), end belongs to year+1.
    Example: year=2019, start=-11-01, end=-03-31 => 2019-11-01 to 2020-03-31
    """
    start_month = int(start_sfx.split("-")[1])
    start_day   = int(start_sfx.split("-")[2])
    end_month   = int(end_sfx.split("-")[1])
    end_day     = int(end_sfx.split("-")[2])

    start = ee.Date.fromYMD(year, start_month, start_day)
    # cross-year?
    end_year = ee.Number(year).add(ee.Number(1) if (end_month, end_day) < (start_month, start_day) else 0)
    end = ee.Date.fromYMD(end_year, end_month, end_day).advance(1, "day")
    return start, end

def ndvi_max_for_season(year, start_sfx, end_sfx, region_geom):
    start, end = season_dates_for_year(year, start_sfx, end_sfx)

    l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterDate(start, end)
          .filterBounds(region_geom)
          .map(preprocess_l8l9_sr))

    l9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
          .filterDate(start, end)
          .filterBounds(region_geom)
          .map(preprocess_l8l9_sr))

    return l8.merge(l9).max().rename(f"NDVImax_{year}")

# -------------------------
# Point-based sampling (fast)
# -------------------------
def to_centroid(feat):
    return ee.Feature(feat.geometry().centroid(1), feat.toDictionary())

def add_ndvi_at_point(stack_img):
    def _fn(feat):
        samp = stack_img.sample(
            region=feat.geometry(),
            scale=SCALE,
            numPixels=1,
            geometries=False,
            tileScale=4
        ).first()

        n19 = ee.Algorithms.If(samp, ee.Feature(samp).get("NDVImax_2019"), None)
        n23 = ee.Algorithms.If(samp, ee.Feature(samp).get("NDVImax_2023"), None)

        return feat.set({"NDVImax_2019": n19, "NDVImax_2023": n23})
    return _fn

def relabel(feat):
    irr = ee.Number(feat.get(LABEL_FIELD))
    n19 = ee.Number(feat.get("NDVImax_2019"))
    n23 = ee.Number(feat.get("NDVImax_2023"))
    diff = n19.subtract(n23).abs()

    # same logic as before: keep irrigated if within tolerance
    new_if_irr = ee.Algorithms.If(diff.lte(NDVI_DIFF_TOL), 1, 0)

    # non-irrigated stays 0 (extend with your rainfed rules if needed)
    new_if_non = 0

    new_label = ee.Number(ee.Algorithms.If(irr.eq(1), new_if_irr, new_if_non))
    return feat.set({"label_new": new_label, "ndvi_diff_abs": diff})

# -------------------------
# Run per ADM1
# -------------------------
tasks = []

for key, (start_sfx, end_sfx) in IRRIG_WINDOW.items():
    # key like "Argentina_Cordoba" or "Brazil_Para"
    parts = key.split("_", 1)
    country = parts[0]
    prov = parts[1]

    # Map our "Argentina"/"Brazil" to GAUL ADM0 country names (may differ slightly)
    # Common GAUL country names: "Argentina", "Brazil"
    gaul_country = country

    geom = get_adm1_geom(gaul_country, prov)
    # If province name doesn't match GAUL exactly, geom will be None.
    # We'll skip safely.
    if ee.Algorithms.IsEqual(geom, None).getInfo():
        print("Skipping (no GAUL match):", key)
        continue

    geom = ee.Geometry(geom)

    # Filter GTPs to province
    fc = gtp_fc.filterBounds(geom).map(to_centroid)

    # Build NDVI stack using province season window
    ndvi2019 = ndvi_max_for_season(2019, start_sfx, end_sfx, geom).rename("NDVImax_2019")
    ndvi2023 = ndvi_max_for_season(2023, start_sfx, end_sfx, geom).rename("NDVImax_2023")
    stack = ndvi2019.addBands(ndvi2023).clip(geom)

    # Sample NDVI values at points, relabel
    fc = (fc.map(add_ndvi_at_point(stack))
            .filter(ee.Filter.notNull(["NDVImax_2019", "NDVImax_2023"]))
            .map(relabel))

    # Export
    desc = f"GMIE_SOAM_{country}_{prov}".replace(" ", "_")
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=desc,
        folder=EXPORT_FOLDER,
        fileFormat="SHP"
    )
    task.start()
    tasks.append(task)
    print("Started:", desc)

print("Total exports started:", len(tasks))


Started: GMIE_SOAM_Argentina_Buenos_Aires
Started: GMIE_SOAM_Argentina_Buenos_Aires_D.f.
Started: GMIE_SOAM_Argentina_Cordoba
Started: GMIE_SOAM_Argentina_Santa_Fe
Started: GMIE_SOAM_Argentina_Entre_Rios
Started: GMIE_SOAM_Argentina_Corrientes
Started: GMIE_SOAM_Argentina_Chaco
Started: GMIE_SOAM_Argentina_Formosa
Started: GMIE_SOAM_Argentina_Misiones
Started: GMIE_SOAM_Argentina_Santiago_Del_Estero
Started: GMIE_SOAM_Argentina_La_Pampa
Skipping (no GAUL match): Argentina_San_Luis
Skipping (no GAUL match): Argentina_Rio_Negro
Started: GMIE_SOAM_Argentina_Tucuman
Started: GMIE_SOAM_Argentina_Jujuy
Started: GMIE_SOAM_Argentina_Salta
Started: GMIE_SOAM_Argentina_Mendoza
Started: GMIE_SOAM_Argentina_San_Juan
Started: GMIE_SOAM_Argentina_La_Rioja
Started: GMIE_SOAM_Argentina_Catamarca
Started: GMIE_SOAM_Argentina_Neuquen
Started: GMIE_SOAM_Argentina_Chubut
Started: GMIE_SOAM_Argentina_Santa_Cruz
Started: GMIE_SOAM_Argentina_Tierra_Del_Fuego
Started: GMIE_SOAM_Brazil_Rondonia
Started: GMIE_S